# Day 19：Function Calling / Tool Use 重点日：工具协议与纯 API 调用闭环（Jupyter Notebook 实战手册）

> **所属阶段**：第 2 阶段 · 第二阶段：大模型核心原理、Prompt工程与全栈 Web API ｜ **第 3 周**：第3周：大模型底层原理、Prompt 工程与工具调用实战  
> **GitLab 工单**：`STARMIND-019` ｜ **开发分支**：`feat/day19-function-calling-loop`  
> **配套 HTML5 课件**：`courseware/day19/index.html` ｜ **独立版本快照**：`versions/day19/`

---

## 🎙️ 老陈开场旁白

早啊各位！今天是整个第二阶段最核心、含金量最高的**重点日**——Function Calling（工具调用，也叫 Tool Use）。为什么说它是 Agent 智能体的灵魂？因为没有工具调用，大模型顶多是个「嘴炮王者」；有了 Function Calling，它就能查实时天气、调高精度计算器、查公司 ERP 数据库、发钉钉告警！注意一个极其关键的底层真相：**大模型自己并不执行你的 Python 函数！**模型只是根据你提供的 JSON Schema，输出一段结构化的 `tool_calls` 申请书，真正执行代码的是咱们的后端循环，执行完把结果封装成 `role='tool'` 喂回给模型，由模型汇总成自然语言。今天咱们不用任何重型框架，纯手写一套 100% 遵循 OpenAI Tool Protocol 的工具注册中心与多步闭环引擎！

### 承上启下
- **昨日基线**：昨天（Day 18）我们用 PromptGuard 守住了安全底线，用 CoT + JSON Mode 实现了高精度意图分类。但林产品提出一个直击灵魂的痛点：纯文本大模型既不知道今天杭州下不下雨，算 18750 * 0.13 税额偶尔会算错，也不知道咱们公司最新的报销差旅标准——它被困在训练截止日期的「鱼缸」里出不来！
- **明日预告**：明天（Day 20）我们将拓展大模型的「眼睛与语义记忆」：学习多模态图像理解/生成 API 封装，并手写 Embedding 向量化与余弦相似度语义匹配工具。


In [ ]:
# 环境自检与路径初始化：确保可以无缝导入当日版本模块并支持异步协程运行
import os
import sys
from pathlib import Path

# 允许在 Jupyter 已有事件循环内安全调用 asyncio.run()
try:
    import nest_asyncio
    nest_asyncio.apply()
except ImportError:
    pass

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'versions').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DAY_DIR = PROJECT_ROOT / 'versions' / 'day19'
for _p in (DAY_DIR, DAY_DIR / 'src', PROJECT_ROOT):
    if _p.exists() and str(_p) not in sys.path:
        sys.path.insert(0, str(_p))
os.chdir(DAY_DIR)
__file__ = str(DAY_DIR / 'src' / 'notebook_entry.py')

print(f'✅ Python 版本: {sys.version.split()[0]}')
print(f'✅ 项目根目录: {PROJECT_ROOT}')
print(f'✅ 当日版本目录: {DAY_DIR}')


## 📋 企业需求背景与验收目标（STARMIND-019）

星辰智脑内部助手需要能够回答员工的实时办公与计算需求，例如「今天杭州天气怎么样、顺便算一下 12800 元不含税金额（税率 6%）以及查一下杭州出差住宿报销上限」。单靠大模型静态记忆会产生严重幻觉。

**今日迭代目标：**
- 实现符合 OpenAI Tools 规范的 `ToolRegistry` 工具注册中心，内置 `get_weather`、`safe_calculator`、`search_internal_kb` 三大企业工具。
- 使用 Python `ast` 模块实现防代码注入的安全数学计算器，彻底杜绝 `eval()` RCE 漏洞。
- 实现完整的 Function Calling 自动多步执行闭环（含并行多工具调用、`tool_call_id` 精确匹配、最大轮次熔断与轨迹审计）。

**验收标准：**
- `export_openai_tools()` 导出的 Schema 100% 符合 OpenAI `type='function'` 标准 JSON Schema 格式。
- `safe_calculator` 能精确计算加减乘除与括号表达式，且对 `__import__('os').system(...)` 等恶意表达式直接拦截抛错。
- 复合问题（同时触发天气 + 计算 + 知识库查询）能在一轮或多轮闭环内完整执行所有工具并整合出准确最终答案。

---

## 📚 课堂核心技术笔记

### 1. 1. OpenAI 标准 Tool Schema 写法规范

一个标准的 Function Calling 定义必须包含三要素：`name`（蛇形命名，如 `get_weather`）、`description`（告诉模型**什么时候**该调它，极其重要！）、`parameters`（标准 JSON Schema，明确 `type`、`properties`、每个参数的 `description` 以及 `required` 必填列表）。如果参数只有固定几个枚举值，一定要加上 `enum: [...]`，能把参数幻觉率降到几乎为零。

> **⚠️ 老陈避坑**：常见坑：`description` 写得太敷衍（比如只写「查天气」），导致模型分不清该传「杭州」还是「杭州市西湖区」。建议在参数描述里直接给示例，如「城市中文名，例如：杭州、北京、上海」。

### 2. 2. 并行工具调用（Parallel Tool Calling）与 tool_call_id 匹配

现代模型（DeepSeek-V3、Qwen2.5、GPT-4o）支持在单次回复中同时返回多个 `tool_calls`（例如同时查杭州和北京天气）。后端必须为**每一个** `tool_call.id` 逐一追加对应的 `role='tool'` 消息；哪怕其中某个工具执行失败，也要把错误说明放进 `content` 回传，绝不能漏掉任何一个 `tool_call_id`，否则下一次 API 请求会被直接拒收（HTTP 400）。

> **⚠️ 老陈避坑**：生产必踩坑：只回传了成功的工具结果，丢弃了报错的工具 `tool_call_id`，导致接口报 `An assistant message with 'tool_calls' must be followed by tool messages responding to each 'tool_call_id'`。

### 3. 3. 工具执行安全：为什么绝对禁止 `eval()`？

很多教程教写计算器工具直接一行 `eval(expression)`，这在企业生产环境等于直接把服务器 Root Shell 送给黑客！一旦发生提示词注入，模型可能传入 `__import__('os').popen('cat /etc/passwd').read()`。必须使用 `ast.parse` 递归白名单求值，只放行 `ast.Constant`、`ast.BinOp`、`ast.UnaryOp` 及加减乘除运算符。

> **⚠️ 老陈避坑**：还要防范拒绝服务攻击（DoS）：比如恶意表达式 `9 ** 9 ** 9` 会瞬间把 CPU 和内存打满，因此对幂运算 `**` 的指数必须做上限拦截（如指数绝对值 `<= 100`）。


## 💻 实操步骤 1：步骤一：实现 OpenAI 规范 ToolRegistry 与天气/AST计算器/内部知识库三大工具

- **目标文件**：`versions/day19/src/day19_tool_registry.py`
- **老陈代码旁白**：咱们先写 `src/day19_tool_registry.py`，把工具定义规范、AST 防注入安全计算器、天气查询和星辰数科内部知识库检索封装进统一的 `ToolRegistry`，并支持一键导出 OpenAI 标准 `tools` Schema 列表。


In [ ]:
"""
StarMind Day 19 - 步骤一：OpenAI 规范 Tool Schema 注册中心与企业三大基础工具
包含：get_weather（天气查询）、safe_calculator（AST安全计算器）、search_internal_kb（内部知识库）。
"""

import ast
import json
import operator
from dataclasses import dataclass
from typing import Any, Callable, Dict, List


# AST 安全算术白名单映射
_BIN_OPS = {
    ast.Add: operator.add,
    ast.Sub: operator.sub,
    ast.Mult: operator.mul,
    ast.Div: operator.truediv,
    ast.Mod: operator.mod,
    ast.Pow: operator.pow,
}
_UNARY_OPS = {
    ast.UAdd: operator.pos,
    ast.USub: operator.neg,
}


def safe_eval_math(expression: str) -> float:
    """
    基于 AST 抽象语法树白名单的安全数学表达式求值器。
    彻底拒绝函数调用、属性访问、变量注入，并对超大幂运算做防 DoS 拦截。
    """
    if not expression or len(expression) > 120:
        raise ValueError("数学表达式为空或过长（上限 120 字符）")

    tree = ast.parse(expression, mode="eval")

    def _eval_node(node: ast.AST) -> float:
        if isinstance(node, ast.Expression):
            return _eval_node(node.body)
        if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)) and not isinstance(node.value, bool):
            return float(node.value)
        if isinstance(node, ast.UnaryOp) and type(node.op) in _UNARY_OPS:
            return _UNARY_OPS[type(node.op)](_eval_node(node.operand))
        if isinstance(node, ast.BinOp) and type(node.op) in _BIN_OPS:
            left = _eval_node(node.left)
            right = _eval_node(node.right)
            if isinstance(node.op, ast.Pow) and abs(right) > 20:
                raise ValueError("安全拦截：幂运算指数绝对值不允许超过 20")
            if isinstance(node.op, ast.Div) and right == 0:
                raise ZeroDivisionError("除数不能为 0")
            return _BIN_OPS[type(node.op)](left, right)
        raise ValueError(f"安全拦截：不允许的表达式节点 {type(node).__name__}")

    return round(_eval_node(tree), 6)


def get_weather(city: str) -> Dict[str, Any]:
    """查询指定城市实时天气（离线仿真企业气象网关数据）。"""
    weather_db = {
        "杭州": {"city": "杭州", "weather": "多云转小雨", "temp_c": 18, "advice": "建议携带折叠伞"},
        "北京": {"city": "北京", "weather": "晴朗微风", "temp_c": 11, "advice": "早晚温差大注意保暖"},
        "上海": {"city": "上海", "weather": "阴天", "temp_c": 19, "advice": "体感舒适"},
        "深圳": {"city": "深圳", "weather": "晴间多云", "temp_c": 26, "advice": "适合短袖出行"},
    }
    clean_city = city.strip().replace("市", "")
    if clean_city in weather_db:
        return {"status": "ok", **weather_db[clean_city]}
    return {"status": "not_found", "city": city, "message": f"暂未收录城市 {city} 的实时气象数据"}


def safe_calculator(expression: str) -> Dict[str, Any]:
    """执行精确数学计算。"""
    result = safe_eval_math(expression)
    return {"status": "ok", "expression": expression, "result": result}


def search_internal_kb(keyword: str) -> Dict[str, Any]:
    """检索杭州星辰数科内部规章制度知识库。"""
    kb_docs = [
        {"id": "HR-001", "title": "差旅住宿报销标准", "content": "一线及新一线城市（北京/上海/深圳/杭州）住宿上限 600 元/晚，其余城市 400 元/晚，需提供增值税电子专票。"},
        {"id": "IT-002", "title": "内网 VPN 申请流程", "content": "新员工入职后在 OA 发起『IT权限申请』，直属主管审批后 30 分钟内自动开通 WireGuard VPN。"},
        {"id": "FIN-003", "title": "研发云资源预算规范", "content": "单项目调用大模型 API 月度预算超过 2000 元需提交架构委员会评审并开启 Prompt 缓存。"},
    ]
    kw = keyword.strip().lower()
    hits = [
        doc for doc in kb_docs
        if kw in doc["title"].lower() or kw in doc["content"].lower() or any(ch in doc["content"] for ch in kw.split() if len(ch) >= 2)
    ]
    return {"status": "ok", "keyword": keyword, "hits": hits, "count": len(hits)}


@dataclass
class ToolSpec:
    name: str
    description: str
    parameters: Dict[str, Any]
    handler: Callable[..., Dict[str, Any]]

    def to_openai_schema(self) -> Dict[str, Any]:
        """导出 100% 符合 OpenAI Chat Completions tools 规范的 Schema 字典。"""
        return {
            "type": "function",
            "function": {
                "name": self.name,
                "description": self.description,
                "parameters": self.parameters,
            },
        }


class ToolRegistry:
    """星辰智脑统一工具注册与安全调度中心。"""

    def __init__(self) -> None:
        self._tools: Dict[str, ToolSpec] = {}

    def register(self, spec: ToolSpec) -> None:
        self._tools[spec.name] = spec

    def export_openai_tools(self) -> List[Dict[str, Any]]:
        return [spec.to_openai_schema() for spec in self._tools.values()]

    def execute(self, name: str, raw_arguments: str | Dict[str, Any]) -> str:
        """
        安全执行指定工具：
        - 自动反序列化 JSON 字符串参数
        - 校验必填参数
        - 捕获一切执行异常并包装为结构化错误 JSON，绝不中断主循环
        """
        if name not in self._tools:
            return json.dumps({"status": "error", "error": f"未注册的工具: {name}"}, ensure_ascii=False)

        spec = self._tools[name]
        try:
            args = json.loads(raw_arguments) if isinstance(raw_arguments, str) else dict(raw_arguments)
            required = spec.parameters.get("required", [])
            missing = [r for r in required if r not in args]
            if missing:
                return json.dumps({"status": "error", "error": f"缺少必填参数: {missing}"}, ensure_ascii=False)
            result = spec.handler(**args)
            return json.dumps(result, ensure_ascii=False)
        except Exception as exc:
            return json.dumps({"status": "error", "error": str(exc)}, ensure_ascii=False)


def build_default_registry() -> ToolRegistry:
    """装配星辰智脑默认三大业务工具。"""
    reg = ToolRegistry()
    reg.register(
        ToolSpec(
            name="get_weather",
            description="查询指定中国城市的实时天气与出行建议。",
            parameters={
                "type": "object",
                "properties": {
                    "city": {"type": "string", "description": "城市中文名，例如：杭州、北京、上海、深圳"},
                },
                "required": ["city"],
            },
            handler=get_weather,
        )
    )
    reg.register(
        ToolSpec(
            name="safe_calculator",
            description="执行高精度四则运算与括号表达式计算（适用于金额、税率、折扣核算）。",
            parameters={
                "type": "object",
                "properties": {
                    "expression": {"type": "string", "description": "纯数学表达式，例如：(12800 / 1.06) * 0.06"},
                },
                "required": ["expression"],
            },
            handler=safe_calculator,
        )
    )
    reg.register(
        ToolSpec(
            name="search_internal_kb",
            description="检索杭州星辰数科内部规章制度（差旅报销、VPN申请、预算规范）。",
            parameters={
                "type": "object",
                "properties": {
                    "keyword": {"type": "string", "description": "检索关键词，例如：差旅、VPN、预算"},
                },
                "required": ["keyword"],
            },
            handler=search_internal_kb,
        )
    )
    return reg


if __name__ == "__main__":
    registry = build_default_registry()
    print("导出的 OpenAI Tools Schema 数量:", len(registry.export_openai_tools()))
    print("天气测试:", registry.execute("get_weather", '{"city": "杭州"}'))
    print("计算测试:", registry.execute("safe_calculator", '{"expression": "600 * 3 + 120"}'))
    print("知识库测试:", registry.execute("search_internal_kb", '{"keyword": "差旅"}'))


### 🔍 步骤 1 代码深度解析

1. `safe_eval_math` 通过 `ast.parse(..., mode='eval')` 白名单求值，既能精确计算嵌套括号和浮点数，又能彻底免疫代码注入与超大幂运算 DoS。
2. `ToolRegistry.execute` 把任何参数缺失或工具运行异常统一封装成 `{"status": "error", ...}` JSON 字符串，保障上游闭环引擎永远不会因为单个工具报错而崩溃。
3. `export_openai_tools` 产出的结构可直接原封不动传给 `openai.OpenAI().chat.completions.create(tools=...)`。

**预期运行输出参考：**
```text
导出的 OpenAI Tools Schema 数量: 3
天气测试: {"status": "ok", "city": "杭州", "weather": "多云转小雨", "temp_c": 18, "advice": "建议携带折叠伞"}
计算测试: {"status": "ok", "expression": "600 * 3 + 120", "result": 1920.0}
```


## 💻 实操步骤 2：步骤二：实现纯 API Function Calling 多步闭环引擎（支持并行调用与熔断）

- **目标文件**：`versions/day19/src/day19_tool_loop_agent.py`
- **老陈代码旁白**：接下来写 `src/day19_tool_loop_agent.py`！这里我们模拟了与 OpenAI `chat.completions.create` 完全一致的响应协议，实现 `user -> model(tool_calls) -> execute -> role='tool' -> model(final_answer)` 的完整自动闭环，并记录每一步的执行轨迹供调试与审计。


In [ ]:
"""
StarMind Day 19 - 步骤二：纯 API Function Calling 多步工具调用闭环引擎
完整实现 OpenAI tool_calls -> 本地执行 -> role="tool" 回填 -> 最终汇总的标准 Agent 闭环。
"""

import json
import re
import uuid
from dataclasses import dataclass, field
from typing import Any, Dict, List

from src.day19_tool_registry import ToolRegistry, build_default_registry


@dataclass
class LoopRunResult:
    """工具调用闭环运行结果与完整审计轨迹。"""
    final_answer: str
    rounds: int
    tool_invocations: List[Dict[str, Any]] = field(default_factory=list)
    messages: List[Dict[str, Any]] = field(default_factory=list)
    stopped_by_limit: bool = False


class MockToolCallingModel:
    """
    离线仿真支持 Parallel Tool Calling 的大模型决策器。
    - 第一轮：解析用户自然语言中的多意图，生成标准 OpenAI `tool_calls` 列表。
    - 后续轮：读取 `role='tool'` 的全部观察结果，汇总生成最终企业级答复。
    """

    _MATH_EXPR_RE = re.compile(r"[\d\.\(\)\+\-\*/\s]{3,}")

    def create_completion(
        self,
        messages: List[Dict[str, Any]],
        tools: List[Dict[str, Any]],
    ) -> Dict[str, Any]:
        # 检查是否已经拿到工具观察结果
        tool_msgs = [m for m in messages if m.get("role") == "tool"]
        if tool_msgs:
            summary_lines = ["星辰智脑已为您完成实时工具核验，汇总结果如下："]
            for tm in tool_msgs:
                data = json.loads(tm["content"])
                t_name = tm.get("name", "unknown")
                if t_name == "get_weather" and data.get("status") == "ok":
                    summary_lines.append(
                        f"- 【天气】{data['city']}：{data['weather']}，{data['temp_c']}°C（{data['advice']}）。"
                    )
                elif t_name == "safe_calculator" and data.get("status") == "ok":
                    summary_lines.append(
                        f"- 【计算结果】`{data['expression']}` = **{data['result']}**。"
                    )
                elif t_name == "search_internal_kb" and data.get("status") == "ok":
                    hits = data.get("hits", [])
                    if hits:
                        summary_lines.append(f"- 【内部制度】{hits[0]['title']}：{hits[0]['content']}")
                    else:
                        summary_lines.append(f"- 【内部制度】未检索到与「{data.get('keyword')}」匹配的制度。")
                else:
                    summary_lines.append(f"- 【工具异常({t_name})】{data.get('error', '执行失败')}")
            return {"role": "assistant", "content": "\n".join(summary_lines), "tool_calls": []}

        # 第一轮：根据最新 user 消息决定需要并行调用的工具集合
        user_text = next((m["content"] for m in reversed(messages) if m.get("role") == "user"), "")
        tool_calls: List[Dict[str, Any]] = []

        # 1. 检测天气需求
        for city in ("杭州", "北京", "上海", "深圳"):
            if city in user_text and ("天气" in user_text or "下雨" in user_text or "温度" in user_text):
                tool_calls.append(self._make_call("get_weather", {"city": city}))

        # 2. 检测数学计算需求
        math_matches = [m.strip() for m in self._MATH_EXPR_RE.findall(user_text) if any(op in m for op in "+-*/")]
        if math_matches:
            tool_calls.append(self._make_call("safe_calculator", {"expression": math_matches[0]}))

        # 3. 检测企业内部知识库需求
        for kb_kw in ("差旅", "报销", "住宿", "VPN", "预算"):
            if kb_kw.lower() in user_text.lower():
                tool_calls.append(self._make_call("search_internal_kb", {"keyword": kb_kw}))
                break

        if tool_calls:
            return {"role": "assistant", "content": None, "tool_calls": tool_calls}

        return {
            "role": "assistant",
            "content": f"您好！您的问题「{user_text}」无需调用外部工具，星辰智脑直接为您解答：如需查询天气、计算报销或检索公司制度，随时告诉我。",
            "tool_calls": [],
        }

    @staticmethod
    def _make_call(name: str, args: Dict[str, Any]) -> Dict[str, Any]:
        return {
            "id": f"call_{uuid.uuid4().hex[:8]}",
            "type": "function",
            "function": {
                "name": name,
                "arguments": json.dumps(args, ensure_ascii=False),
            },
        }


class FunctionCallingLoopEngine:
    """标准纯 API Function Calling 多步执行闭环引擎。"""

    def __init__(
        self,
        registry: ToolRegistry | None = None,
        model: Any | None = None,
        max_rounds: int = 5,
    ) -> None:
        self.registry = registry or build_default_registry()
        self.model = model or MockToolCallingModel()
        self.max_rounds = max_rounds

    def run(self, user_query: str, system_prompt: str = "你是星辰智脑企业办公助手，请善用工具精准回答。") -> LoopRunResult:
        messages: List[Dict[str, Any]] = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_query},
        ]
        tools_schema = self.registry.export_openai_tools()
        invocations: List[Dict[str, Any]] = []

        for round_idx in range(1, self.max_rounds + 1):
            assistant_msg = self.model.create_completion(messages, tools_schema)
            messages.append(assistant_msg)
            tool_calls = assistant_msg.get("tool_calls") or []

            if not tool_calls:
                return LoopRunResult(
                    final_answer=str(assistant_msg.get("content") or ""),
                    rounds=round_idx,
                    tool_invocations=invocations,
                    messages=messages,
                    stopped_by_limit=False,
                )

            # 严格按照 OpenAI 协议：逐一执行每个 tool_call 并回填对应 tool_call_id
            for call in tool_calls:
                call_id = call["id"]
                fn_name = call["function"]["name"]
                fn_args = call["function"]["arguments"]
                tool_output = self.registry.execute(fn_name, fn_args)

                invocations.append({
                    "round": round_idx,
                    "tool_call_id": call_id,
                    "name": fn_name,
                    "arguments": fn_args,
                    "output": tool_output,
                })
                messages.append({
                    "role": "tool",
                    "tool_call_id": call_id,
                    "name": fn_name,
                    "content": tool_output,
                })

        return LoopRunResult(
            final_answer="⚠️ 工具调用轮次达到安全上限，已触发熔断机制，请简化问题后重试。",
            rounds=self.max_rounds,
            tool_invocations=invocations,
            messages=messages,
            stopped_by_limit=True,
        )


if __name__ == "__main__":
    engine = FunctionCallingLoopEngine()
    query = "我明天去杭州出差，帮我查下杭州天气、公司差旅住宿标准，顺便算一下 600 * 3 + 240 是多少钱？"
    res = engine.run(query)
    print(f"闭环轮次: {res.rounds} | 调用工具数: {len(res.tool_invocations)}")
    print(res.final_answer)


### 🔍 步骤 2 代码深度解析

1. `FunctionCallingLoopEngine.run` 的循环体几乎和真实生产中调用 DeepSeek / OpenAI SDK 的代码 1:1 同构——只需把 `self.model.create_completion(...)` 换成 `openai_client.chat.completions.create(...)` 即可无缝上线。
2. 支持单次并行触发多个工具（本例一次触发天气 + 计算器 + 内部知识库 3 个工具），并保证每个 `tool_call_id` 都有对应的 `role='tool'` 消息回填。
3. 循环外兜底 `stopped_by_limit=True`，实现生产级熔断保护。

**预期运行输出参考：**
```text
闭环轮次: 2 | 调用工具数: 3
星辰智脑已为您完成实时工具核验，汇总结果如下：
- 【天气】杭州：多云转小雨，18°C（建议携带折叠伞）。
- 【计算结果】`600 * 3 + 240` = **2040.0**。
- 【内部制度】差旅住宿报销标准：一线及新一线城市（北京/上海/深圳/杭州）住宿上限 600 元/晚...
```


## 🧪 当日自动化单元测试

在终端运行 `pytest versions/day19/tests/test_day19.py -v` 即可验收全部测试断言。下面展示完整测试脚本源码，供对照学习测试写法：

```python
"""Day 19 自动化单元测试：验证 Tool Schema、AST 安全防注入与多步 Function Calling 闭环。"""

import json
import pytest
from src.day19_tool_registry import build_default_registry, safe_eval_math
from src.day19_tool_loop_agent import FunctionCallingLoopEngine


def test_openai_tool_schema_and_safe_calculator():
    reg = build_default_registry()
    schemas = reg.export_openai_tools()
    assert len(schemas) == 3
    for s in schemas:
        assert s["type"] == "function"
        assert "name" in s["function"]
        assert "parameters" in s["function"]

    assert safe_eval_math("(600 * 3 + 240) / 2") == pytest.approx(1020.0)
    # 验证安全拦截代码注入与超大幂 DoS
    with pytest.raises(ValueError):
        safe_eval_math("__import__('os').system('echo hacked')")
    with pytest.raises(ValueError):
        safe_eval_math("9 ** 999")


def test_tool_registry_error_isolation():
    reg = build_default_registry()
    err_json = reg.execute("safe_calculator", '{"expression": "10 / 0"}')
    data = json.loads(err_json)
    assert data["status"] == "error"
    assert "0" in data["error"]

    unknown_json = reg.execute("drop_database", "{}")
    assert json.loads(unknown_json)["status"] == "error"


def test_parallel_function_calling_loop():
    engine = FunctionCallingLoopEngine()
    res = engine.run("查一下杭州天气和公司差旅报销标准，并算下 600 * 3 + 150")
    assert res.rounds == 2
    assert res.stopped_by_limit is False
    assert len(res.tool_invocations) == 3
    called_names = {inv["name"] for inv in res.tool_invocations}
    assert called_names == {"get_weather", "safe_calculator", "search_internal_kb"}
    assert "1950.0" in res.final_answer
    assert "杭州" in res.final_answer
    assert "600 元/晚" in res.final_answer


def test_loop_circuit_breaker_max_rounds():
    class InfiniteLoopModel:
        def create_completion(self, messages, tools):
            return {
                "role": "assistant",
                "content": None,
                "tool_calls": [
                    {"id": "call_loop", "type": "function", "function": {"name": "get_weather", "arguments": '{"city":"杭州"}'}}
                ],
            }

    engine = FunctionCallingLoopEngine(model=InfiniteLoopModel(), max_rounds=3)
    res = engine.run("测试熔断")
    assert res.stopped_by_limit is True
    assert res.rounds == 3

```


## 📝 课后梯度实战作业与标准答案

建议先在下方新建空白单元格独立完成编码，运行通过后再核对参考答案。


### 作业 1（基础巩固）：为 ToolRegistry 新增企业货币汇率换算工具

**题目要求：**
编写 `convert_currency(amount, from_currency, to_currency)` 函数（内置 CNY/USD/EUR/JPY 静态参考汇率），注册到 `ToolRegistry` 中并在 Schema 里用 `enum` 限制币种，完成调用测试。

**思路提示：** 统一先换算为 CNY 基准金额，再除以目标币种汇率，保留 2 位小数。


In [ ]:
import json
from src.day19_tool_registry import ToolSpec, build_default_registry

RATES_TO_CNY = {"CNY": 1.0, "USD": 7.25, "EUR": 7.85, "JPY": 0.048}


def convert_currency(amount: float, from_currency: str, to_currency: str):
    if from_currency not in RATES_TO_CNY or to_currency not in RATES_TO_CNY:
        raise ValueError("不支持的币种")
    cny = amount * RATES_TO_CNY[from_currency]
    converted = round(cny / RATES_TO_CNY[to_currency], 2)
    return {"status": "ok", "amount": amount, "from": from_currency, "to": to_currency, "converted": converted}


if __name__ == "__main__":
    reg = build_default_registry()
    reg.register(
        ToolSpec(
            name="convert_currency",
            description="进行常用企业外币汇率换算",
            parameters={
                "type": "object",
                "properties": {
                    "amount": {"type": "number"},
                    "from_currency": {"type": "string", "enum": list(RATES_TO_CNY)},
                    "to_currency": {"type": "string", "enum": list(RATES_TO_CNY)},
                },
                "required": ["amount", "from_currency", "to_currency"],
            },
            handler=convert_currency,
        )
    )
    print(reg.execute("convert_currency", json.dumps({"amount": 100, "from_currency": "USD", "to_currency": "CNY"})))


**答案解析与评分标准：**

在 Schema 中使用 `enum` 约束离散枚举值，可以大幅减少模型传参拼写错误（如把 `CNY` 写成 `RMB`）。


### 作业 2（企业实战）：Python 函数签名自动转 OpenAI Tool Schema 装饰器

**题目要求：**
使用标准库 `inspect.signature` 和类型注解编写 `function_to_tool_schema(func)`，自动把一个带类型注解和 docstring 的 Python 函数转换为 OpenAI `type='function'` Schema 字典。

**思路提示：** 映射 `str -> 'string'`、`int -> 'integer'`、`float -> 'number'`、`bool -> 'boolean'`，无默认值的参数加入 `required`。


In [ ]:
import inspect
from typing import Any, Callable, Dict

PY_TO_JSON_TYPE = {str: "string", int: "integer", float: "number", bool: "boolean"}


def function_to_tool_schema(func: Callable[..., Any]) -> Dict[str, Any]:
    """根据函数签名与类型注解自动生成 OpenAI Function Calling Schema。"""
    sig = inspect.signature(func)
    properties: Dict[str, Any] = {}
    required = []
    for name, param in sig.parameters.items():
        json_type = PY_TO_JSON_TYPE.get(param.annotation, "string")
        properties[name] = {"type": json_type}
        if param.default is inspect.Parameter.empty:
            required.append(name)
    return {
        "type": "function",
        "function": {
            "name": func.__name__,
            "description": (func.__doc__ or "").strip(),
            "parameters": {"type": "object", "properties": properties, "required": required},
        },
    }


def create_jira_ticket(title: str, priority: int = 2) -> dict:
    """在企业工单系统中创建一条新研发工单。"""
    return {"title": title, "priority": priority}


if __name__ == "__main__":
    print(function_to_tool_schema(create_jira_ticket))


**答案解析与评分标准：**

LangChain 的 `@tool` 装饰器和 FastMCP 底层正是利用 `inspect` + Pydantic 实现自动化 Schema 提取的！


### 作业 3（架构进阶）：高危工具人工确认拦截器（Human-in-the-Loop Gate）

**题目要求：**
为 `ToolRegistry` 包装一层权限守卫：当调用高危工具（如 `approve_refund` 或 `send_mass_email`）时，若 `approval_callback(tool_name, args)` 返回 `False`，则拒绝执行并返回 `{"status": "rejected_by_human"}`。

**思路提示：** 定义一个 `GuardedToolExecutor`，维护 `sensitive_tools: set[str]`，执行前先触发回调函数。


In [ ]:
import json
from typing import Any, Callable, Dict, Set
from src.day19_tool_registry import ToolRegistry, ToolSpec


class GuardedToolExecutor:
    """带人工二次确认（Human-in-the-Loop）的高危工具执行守卫。"""

    def __init__(self, registry: ToolRegistry, sensitive_tools: Set[str]) -> None:
        self.registry = registry
        self.sensitive_tools = sensitive_tools

    def execute_with_approval(
        self,
        name: str,
        arguments: Dict[str, Any],
        approval_cb: Callable[[str, Dict[str, Any]], bool],
    ) -> str:
        if name in self.sensitive_tools and not approval_cb(name, arguments):
            return json.dumps(
                {"status": "rejected_by_human", "tool": name, "message": "操作已被人工审核员拒绝"},
                ensure_ascii=False,
            )
        return self.registry.execute(name, arguments)


if __name__ == "__main__":
    reg = ToolRegistry()
    reg.register(
        ToolSpec(
            name="approve_refund",
            description="自动执行大额退款",
            parameters={"type": "object", "properties": {"order_id": {"type": "string"}}, "required": ["order_id"]},
            handler=lambda order_id: {"status": "refunded", "order_id": order_id},
        )
    )
    guard = GuardedToolExecutor(reg, sensitive_tools={"approve_refund"})
    print("拒绝演示:", guard.execute_with_approval("approve_refund", {"order_id": "ORD-99"}, lambda n, a: False))
    print("批准演示:", guard.execute_with_approval("approve_refund", {"order_id": "ORD-99"}, lambda n, a: True))


**答案解析与评分标准：**

所有涉及资金、发外部邮件、改库删表的 Agent 工具，在生产中都必须采用 Human-in-the-Loop 二次确认机制。


---

## 🎯 老陈收尾复盘

今天把 Function Calling 闭环从头到尾手撸一遍，大家应该彻底打通任督二脉了！哪怕后面学 LangChain、LangGraph、MCP，底层跑的依然是今天这四步曲：`定义 Tool Schema -> 模型返回 tool_calls -> 本地安全执行并回填 role='tool' -> 模型二次总结`。记住两条生产保命原则：第一，计算器绝不能直接用 `eval()`，必须用 `ast` 安全求值；第二，工具循环必须设 `max_steps` 最大步数熔断，防止模型死循环烧光额度！
